In [ ]:
import json
import pandas as pd
import os
import shutil
from glob import glob
import zipfile
import numpy as np
from tqdm import tqdm
from shapely import wkb
from shapely.geometry import shape
import geopandas as gpd
from shapely.geometry import Point

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
QV_THRESHOLD = 20 

In [ ]:
xenium_annotation_list=glob('../../data/spatialTranscriptome/xenium_seg/TENX*_xenium_nucleus_seg.parquet')
xenium_transcripts_list=[f.replace("xenium_seg/", "transcripts/") for f in xenium_annotation_list]
xenium_transcripts_list=[f.replace("_xenium_nucleus_seg.parquet", "_transcripts.parquet") for f in xenium_transcripts_list]

# 고속 RNA 검출 (BLANK|NegControl|antisense 제외)
RnA_unique_set = set()  # list 대신 set 사용 (중복 제거 O(1))

for i in tqdm(range(len(xenium_transcripts_list))):
    df_transcript = pd.read_parquet(xenium_transcripts_list[i], columns=['feature_name', 'overlaps_nucleus'])
    df_transcript = df_transcript[df_transcript['overlaps_nucleus'] == 1]
    
    # bytes -> str 변환 (한 번만)
    if len(df_transcript) > 0:
        if isinstance(df_transcript['feature_name'].iloc[0], bytes):
            feature_names = df_transcript['feature_name'].str.decode('utf-8')
        else:
            feature_names = df_transcript['feature_name']
        
        # 벡터화된 필터링 (정규식 한 번만)
        mask = ~feature_names.str.contains('BLANK|NegControl|antisense', case=False, na=False, regex=True)
        valid_features = feature_names[mask].unique()
        
        RnA_unique_set.update(valid_features)

RnA_unique_list = sorted(list(RnA_unique_set))
print(f"Total unique RNAs: {len(RnA_unique_list)}")



 33%|███▎      | 12/36 [01:52<03:56,  9.84s/it]

In [ ]:
# NCBI 마커 gene list
marker_genes = {

    "Epithelial": [
        "EPCAM", "KRT8", "KRT18", "KRT19", "KRT7",
        "ERBB2", "ESR1", "PGR", "GATA3", "FOXA1",
        "CDH1", "CLDN3", "CLDN4", "CLDN7",
        "KRT5", "KRT14", "KRT15", "KRT17", "TP63",
        "MKI67", "PCNA"
    ],

    "Stromal": [
        "COL1A1", "COL1A2", "COL3A1", "COL5A1", "COL6A1",
        "FN1", "VIM", "DCN", "LUM",
        "PDGFRA", "PDGFRB", "FAP", "S100A4",
        "DPT", "FBLN1", "POSTN", "THY1",
        "ACTA2", "TAGLN", "MYH11", "CNN1", "MYL9",
        "ACTG2", "DES", "SMTN"
    ],

    "Lymphocyte": [
        # T cell
        "CD3D", "CD3E", "CD3G", "TRAC", "TRBC1", "TRBC2",
        "CD4", "CD8A", "CD8B",
        "CCR7", "IL7R", "TCF7", "LEF1",
        # NK
        "NKG7", "GNLY", "KLRD1", "PRF1", "GZMB", "GZMK"
    ],

    "Plasma": [
        "MZB1", "JCHAIN", "SDC1", "XBP1", "PRDM1",
        "TNFRSF17", "SLAMF7",
        "IGHG1", "IGHG2", "IGHG3", "IGHG4",
        "IGHA1", "IGHA2", "IGKC"
    ],

    "Neutrophil": [
        "S100A8", "S100A9", "S100A12",
        "CSF3R", "FCGR3B",
        "MPO", "ELANE", "CTSG",
        "CEACAM8", "CXCR1", "CXCR2",
        "MMP9", "CAMP"
    ],

    "Eosinophil": [
        "SIGLEC8", "IL5RA", "CCR3",
        "PRG2", "PRG3", "PRG4",
        "RNASE2", "RNASE3",
        "CLC", "EPX",
        "CPA3", "HDC", "ALOX15",
        "IL3RA", "ADGRE1"
    ],

    "Other/Unknown": []
}

class_list = {
    0: "Epithelial",
    1: "Stromal",
    2: "Lymphocyte",
    3: "Plasma",
    4: "Neutrophil",
    5: "Eosinophil",
    6: "Other/Unknown"
}

class_colors_hex = {
    "Epithelial": "#FF0000",           # 빨강
    "Stromal": "#00FF00",              # 초록
    "Lymphocyte": "#FFFF00",           # 노랑
    "Plasma": "#FF00FF",               # 마젠타
    "Neutrophil": "#1E90FF",           # DodgerBlue (밝은 파랑)
    "Eosinophil": "#FFA500",           # 오렌지
    "Other/Unknown": "#808080"         # 회색
}

class_colors = {
    "Epithelial": [255, 0, 0],         # 빨강
    "Stromal": [0, 255, 0],            # 초록
    "Lymphocyte": [255, 255, 0],       # 노랑
    "Plasma": [255, 0, 255],           # 마젠타
    "Neutrophil": [30, 144, 255],      # DodgerBlue (밝은 파랑)
    "Eosinophil": [255, 165, 0],       # 오렌지
    "Other/Unknown": [128, 128, 128]   # 회색
}
marker_weights = {

    "Epithelial": {
        "EPCAM": 3.0,
        "KRT8": 2.5,
        "KRT18": 2.5,
        "KRT19": 2.5,
        "KRT7": 2.0,
        "CDH1": 2.0,
        "CLDN3": 1.5,
        "CLDN4": 1.5,
        "CLDN7": 1.5,
        "KRT5": 1.0,
        "KRT14": 1.0,
        "TP63": 1.0,
        "MKI67": 0.5,
        "PCNA": 0.5,
    },

    "Stromal": {
        "COL1A1": 3.0,
        "COL1A2": 3.0,
        "COL3A1": 3.0,
        "DCN": 2.5,
        "LUM": 2.5,
        "PDGFRA": 2.5,
        "PDGFRB": 2.5,
        "FAP": 2.0,
        "FN1": 1.5,
        "VIM": 1.5,
        "S100A4": 1.5,
        "ACTA2": 1.0,
        "TAGLN": 1.0,
        "MYH11": 1.0,
    },

    "Lymphocyte": {
        "CD3D": 3.0,
        "CD3E": 3.0,
        "CD3G": 3.0,
        "TRAC": 3.0,
        "CD4": 2.0,
        "CD8A": 2.0,
        "CD8B": 2.0,
        "CCR7": 1.5,
        "IL7R": 1.5,
        "NKG7": 2.5,
        "GNLY": 2.5,
        "PRF1": 2.0,
        "GZMB": 2.0,
    },

    "Plasma": {
        "MZB1": 3.5,
        "JCHAIN": 3.5,
        "SDC1": 3.0,
        "XBP1": 3.0,
        "PRDM1": 3.0,
        "TNFRSF17": 2.5,
        "IGHG1": 2.5,
        "IGHG2": 2.5,
        "IGHG3": 2.5,
        "IGHG4": 2.5,
        "IGHA1": 2.5,
        "IGHA2": 2.5,
        "IGKC": 2.0,
    },

    "Neutrophil": {
        "S100A8": 3.5,
        "S100A9": 3.5,
        "S100A12": 3.5,
        "CSF3R": 3.0,
        "FCGR3B": 3.0,
        "MPO": 2.5,
        "ELANE": 2.5,
        "CTSG": 2.5,
        "CEACAM8": 2.5,
        "CXCR1": 2.0,
        "CXCR2": 2.0,
    },

    "Eosinophil": {
        "SIGLEC8": 3.5,
        "CPA3": 3.5,
        "IL5RA": 3.0,
        "PRG2": 3.0,
        "PRG3": 2.5,
        "PRG4": 2.0,
        "RNASE2": 2.5,
        "RNASE3": 2.5,
        "EPX": 2.5,
        "CLC": 2.0,
        "HDC": 2.0,
        "ALOX15": 2.0,
        "IL3RA": 1.5,
        "ADGRE1": 1.5,
        "CCR3": 2.0,
    }
}


In [ ]:
# TENX 마커 gene list
marker_genes = {
    "Epithelial": [
        "EPCAM", "KRT8", "KRT18", "KRT7",
        "TACSTD2", "CLDN4",
        "ERBB2", "ESR1", "PGR", "GATA3",
        "MKI67", "PCNA",
        "CDH1",
        "KRT5", "KRT14", "KRT15", "KRT16", "KRT17", "KRT20", "KRT23", "KRT80",
        "MSLN", "WFDC2"
    ],

    "Stromal": [
        "COL1A1", "COL1A2", "COL5A2",
        "COL6A1", "COL6A2", "COL11A1",
        "DCN", "LUM", "SPARC", "FN1",
        "PDGFRA", "FAP", "CXCL12", "DPT",
        "MMP2", "MMP14",
        "C7", "C1R", "C1S", "FBLN1", "POSTN", "THY1",
        "ACTA2", "TAGLN", "MYH11", "CNN1",
        "PDGFRB", "RGS5",
        "KRT5", "KRT14", "KRT15", "KRT17", "TP63",
        "COL17A1", "ITGA6", "LAMC2"
    ],

    "Lymphocyte": [
        # T cell
        "PTPRC", "CD3D", "CD3E", "CD3G",
        "TRAC", "TRBC1", "TRBC2",
        "CD4", "CD8A", "CD8B",
        "IL7R", "CCR7", "LTB",
        # NK
        "NKG7", "GNLY", "PRF1", "GZMK", "GZMB", "GZMH", "KLRD1",
        # B cell
        "MS4A1", "CD79A", "CD79B", "CD19", "CD74",
        # 공통 lymphocyte
        "CD2", "CD5", "CD7", "CD27", "CD28", "CD52", "CD69"
    ],

    "Plasma": [
        "MZB1", "XBP1", "JCHAIN",
        "SDC1", "TNFRSF17",
        "IGKC", "IGLC3",
        "IGHG1", "IGHG3", "IGHG4", "IGHM",
        "DERL3", "SEC11C", "SSR4"
    ],

    "Neutrophil": [
        "S100A8", "S100A9", "S100A12",
        "CXCR2", "CXCR1",
        "CTSG", "LTF", "LCN2",
        "MNDA", "CSF3R"
    ],

    "Eosinophil": [
        # TENX 패널에서 사용 가능한 호산구 마커
        "IL5RA", "CCR3", "PRG2", "PRG3",
        "RNASE2", "RNASE3",
        "CPA3", "EPX", "CLC"
    ],

    "Other/Unknown": []
}

class_list = {
    0: "Epithelial",
    1: "Stromal",
    2: "Lymphocyte",
    3: "Plasma",
    4: "Neutrophil",
    5: "Eosinophil",
    6: "Other/Unknown"
}

class_colors_hex = {
    "Epithelial": "#FF0000",           # 빨강
    "Stromal": "#00FF00",              # 초록
    "Lymphocyte": "#FFFF00",           # 노랑
    "Plasma": "#FF00FF",               # 마젠타
    "Neutrophil": "#1E90FF",           # DodgerBlue (밝은 파랑)
    "Eosinophil": "#FFA500",           # 오렌지
    "Other/Unknown": "#808080"         # 회색
}

class_colors = {
    "Epithelial": [255, 0, 0],         # 빨강
    "Stromal": [0, 255, 0],            # 초록
    "Lymphocyte": [255, 255, 0],       # 노랑
    "Plasma": [255, 0, 255],           # 마젠타
    "Neutrophil": [30, 144, 255],      # 도저블루 (밝은 파랑)
    "Eosinophil": [255, 165, 0],       # 오렌지
    "Other/Unknown": [128, 128, 128]   # 회색
}

marker_weights = {
    "Epithelial": {
        "EPCAM": 3.0,
        "TACSTD2": 3.0,
        "KRT8": 2.5,
        "KRT18": 2.5,
        "KRT7": 2.0,
        "CLDN4": 1.5,
        "CDH1": 2.0,
        "ERBB2": 1.0, "ESR1": 1.0, "PGR": 1.0, "GATA3": 1.0,
        "KRT5": 1.0, "KRT14": 1.0, "KRT15": 1.0, "KRT17": 1.0,
        "KRT16": 0.8, "KRT20": 1.2, "KRT23": 1.0, "KRT80": 1.0,
        "MKI67": 0.5, "PCNA": 0.5,
        "MSLN": 1.5, "WFDC2": 1.5,
    },

    "Stromal": {
        "COL1A1": 3.0, "COL1A2": 3.0,
        "COL5A2": 2.5, "COL6A1": 2.5, "COL6A2": 2.5, "COL11A1": 2.5,
        "DCN": 2.5, "LUM": 2.5,
        "SPARC": 2.0, "FN1": 1.5,
        "PDGFRA": 2.5, "FAP": 2.0, "CXCL12": 2.0, "DPT": 1.5,
        "MMP2": 1.5, "MMP14": 1.5,
        "POSTN": 1.5, "FBLN1": 1.5, "THY1": 1.0,
        "C7": 1.0, "C1R": 1.0, "C1S": 1.0,
        "ACTA2": 1.0, "TAGLN": 1.0, "MYH11": 1.0, "CNN1": 1.0,
        "PDGFRB": 2.0, "RGS5": 1.5,
        # basal overlap는 낮게
        "KRT5": 0.5, "KRT14": 0.5, "KRT15": 0.5, "KRT17": 0.5, "TP63": 0.5,
        "COL17A1": 1.0, "ITGA6": 1.0, "LAMC2": 1.0,
    },

    "Lymphocyte": {
        "PTPRC": 0.5,
        # T cell 마커 (높은 가중치)
        "CD3D": 3.0, "CD3E": 3.0, "CD3G": 3.0, "TRAC": 3.0,
        "TRBC1": 2.0, "TRBC2": 2.0,
        "CD4": 2.0, "CD8A": 2.0, "CD8B": 2.0,
        "IL7R": 1.5, "CCR7": 1.5, "LTB": 1.0,
        # NK 마커 (높은 가중치)
        "NKG7": 3.0, "GNLY": 3.0, "PRF1": 2.5, "GZMB": 2.5, "GZMK": 2.0, "GZMH": 2.0, "KLRD1": 2.0,
        # B cell 마커 (높은 가중치)
        "CD79A": 3.0, "CD79B": 3.0, "MS4A1": 2.5, "CD19": 2.0, "CD74": 1.5,
        # 공통 lymphocyte 마커
        "CD2": 1.0, "CD5": 1.0, "CD7": 1.0, "CD27": 1.0, "CD28": 1.0, "CD52": 0.8, "CD69": 0.8,
    },

    "Plasma": {
        # 플라즈마 특이 마커 (높은 가중치)
        "MZB1": 3.5, "JCHAIN": 3.5, "XBP1": 3.0, 
        "SDC1": 3.0, "TNFRSF17": 2.5,
        # 면역글로불린
        "IGKC": 2.0, "IGLC3": 2.0, 
        "IGHG1": 2.5, "IGHG3": 2.5, "IGHG4": 2.5, "IGHM": 2.0,
        # ER 관련
        "DERL3": 1.0, "SEC11C": 1.0, "SSR4": 1.0,
    },

    "Neutrophil": {
        # 호중구 특이 마커 (높은 가중치)
        "S100A8": 3.5, "S100A9": 3.5, "S100A12": 3.5,
        "CSF3R": 3.0,
        "CXCR1": 2.0, "CXCR2": 2.0,
        "CTSG": 2.5, "LTF": 2.5, "LCN2": 2.0, "MNDA": 2.0,
    },

    "Eosinophil": {
        # TENX 패널에서 사용 가능한 호산구 마커
        "IL5RA": 2.5, "CCR3": 2.5, "PRG2": 3.0, "PRG3": 2.5,
        "RNASE2": 2.5, "RNASE3": 2.5,
        "CPA3": 3.0, "EPX": 2.5, "CLC": 2.0,
    }
}


In [4]:
xenium_annotation_list=glob('../../data/spatialTranscriptome/xenium_seg/NC*_xenium_nucleus_seg.parquet')
xenium_transcripts_list=[f.replace("xenium_seg/", "transcripts/") for f in xenium_annotation_list]
xenium_transcripts_list=[f.replace("_xenium_nucleus_seg.parquet", "_transcripts.parquet") for f in xenium_transcripts_list]


def classify_cell_by_genes(gene_list, marker_dict, marker_weights=None, min_score=2, min_confidence_ratio=1.0):
    """
    여러 유전자를 기반으로 cell type scoring (가중치 반영)
    
    Args:
        gene_list: 셀 내 검출된 유전자 리스트
        marker_dict: cell type별 marker gene 딕셔너리
        marker_weights: cell type별 marker 유전자 가중치 딕셔너리
        min_markers: cell type으로 분류하기 위한 최소 marker 수 (기본값: 2)
        min_confidence_ratio: 2등 대비 1등의 최소 비율 (기본값: 0.8)
    
    Returns:
        cell_type: 분류된 cell type
        scores: 각 cell type별 가중치 점수
    """
    if marker_weights is None:
        marker_weights = {}

    scores = {cell_type: 0.0 for cell_type in marker_dict.keys()}
    counts = {cell_type: 0 for cell_type in marker_dict.keys()}
    
    # 각 gene이 어느 cell type의 marker인지 카운트 + 가중치 합산
    for gene in gene_list:
        for cell_type, markers in marker_dict.items():
            if gene in markers:
                weight = marker_weights.get(cell_type, {}).get(gene, 1.0)
                scores[cell_type] += weight
                counts[cell_type] += 1
    
    # 점수 정렬
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    max_score = sorted_scores[0][1]
    second_score = sorted_scores[1][1] if len(sorted_scores) > 1 else 0
    best_type = sorted_scores[0][0]
    
    # ⚠️ 최소 marker 수 미만이면 Unknown (가중치가 아니라 개수 기준)
    if max_score < min_score:
        return 'Other/Unknown', scores
    
    # ⚠️ 2등과의 차이가 너무 작으면 애매한 경우 -> Unknown
    if second_score > 0 and second_score / max_score >= min_confidence_ratio:
        return 'Other/Unknown', scores
    
    # 명확한 1등
    return best_type, scores

save_path='../../data/spatialTranscriptome/detail_preprocessed_xenium/'
xenium_transcripts_list=[f.replace("xenium_seg/", "transcripts/") for f in xenium_annotation_list]
xenium_transcripts_list=[f.replace("_xenium_nucleus_seg.parquet", "_transcripts.parquet") for f in xenium_transcripts_list]

UNKNOWN_THRESHOLD = 15.0  # Unknown이 15% 이상이면 샘플 폐기

for i in range(len(xenium_transcripts_list)):
    xenium_transcript_path = xenium_transcripts_list[i]
    df_transcript = pd.read_parquet(xenium_transcript_path)
    df_filtered = df_transcript[df_transcript['qv'] > QV_THRESHOLD].copy()
    if type(df_filtered['feature_name'].iloc[0])==bytes:
        df_filtered = df_filtered[~df_filtered['feature_name'].str.decode('utf-8').str.contains('BLANK|NegControl|antisense', case=False, na=False)]
        df_filtered['feature_name'] = df_filtered['feature_name'].str.decode('utf-8').replace("b'", "", regex=False).str.replace("'", "", regex=False)
    elif type(df_filtered['feature_name'].iloc[0])==str:
        df_filtered = df_filtered[~df_filtered['feature_name'].str.contains('BLANK|NegControl|antisense', case=False, na=False)]
    else:
        print(xenium_transcript_path)
        continue
    df_filtered
    xenium_annotation_path=xenium_annotation_list[i]
    df_seg = pd.read_parquet(xenium_annotation_path)
    df=pd.DataFrame(columns=['x1','y1','x2','y2','class_name'])
    annotations = []
    grouped_transcripts = df_filtered.groupby('cell_id')['feature_name'].apply(list).to_dict()
    
    unknown_count = 0  # Unknown 개수 추적
    
    for j in tqdm(range(len(df_seg))):
        temp_df_seg=df_seg.iloc[j]
        cell_id=temp_df_seg.name
        geom_binary=temp_df_seg['geometry']
        polygon = wkb.loads(geom_binary)
        x,y=polygon.exterior.xy
        x1=int(np.min(x))
        y1=int(np.min(y))
        x2=int(np.max(x))
        y2=int(np.max(y))
        try:
            genes_in_cell=grouped_transcripts[cell_id]
            cell_type, score = classify_cell_by_genes(genes_in_cell, marker_genes, marker_weights=marker_weights)
            
            # ⚠️ Unknown은 저장하지 않음 (학습에서 제외)
            if cell_type == 'Other/Unknown':
                unknown_count += 1
                continue
            
            annotations.append({
                'x1': x1,
                'y1': y1,
                'x2': x2,
                'y2': y2,
                'class_name': cell_type,
            })
        except KeyError:
            continue
    
    df = pd.DataFrame(annotations)
    
    # 통계 출력
    total_cells = len(df_seg)
    classified_cells = len(df)
    unknown_ratio = unknown_count / total_cells * 100 if total_cells > 0 else 0
    
    print(f"\n=== {os.path.basename(xenium_annotation_path)} ===")
    print(f"Total cells: {total_cells}")
    print(f"Classified cells: {classified_cells} ({classified_cells/total_cells*100:.1f}%)")
    print(f"Unknown cells (excluded): {unknown_count} ({unknown_ratio:.1f}%)")
    
    # ⚠️ Unknown 비율이 15% 이상이면 샘플 폐기
    if unknown_ratio >= UNKNOWN_THRESHOLD:
        print(f"❌ SKIPPED: Unknown ratio ({unknown_ratio:.1f}%) >= {UNKNOWN_THRESHOLD}% threshold")
        print("   This sample has too many unclassifiable cells and will not be saved.")
        continue
    
    if len(df) > 0:
        print("\nCell type distribution (saved):")
        type_counts = df['class_name'].value_counts()
        for cell_type, count in type_counts.items():
            percentage = (count / len(df)) * 100
            print(f"  {cell_type}: {count} ({percentage:.1f}%)")
    
    create_dir(save_path+'labels/')    
    df.to_csv(save_path+'labels/'+os.path.basename(xenium_annotation_path).replace('_xenium_nucleus_seg.parquet', '.csv'), index=False)
    print(f"✅ SAVED: {os.path.basename(xenium_annotation_path).replace('_xenium_nucleus_seg.parquet', '.csv')}")


100%|██████████| 108131/108131 [00:20<00:00, 5204.39it/s]



=== NCBI864_xenium_nucleus_seg.parquet ===
Total cells: 108131
Classified cells: 106780 (98.8%)
Unknown cells (excluded): 1193 (1.1%)

Cell type distribution (saved):
  Stromal: 90471 (84.7%)
  Plasma: 7128 (6.7%)
  Epithelial: 5359 (5.0%)
  Lymphocyte: 2198 (2.1%)
  Neutrophil: 1624 (1.5%)
✅ SAVED: NCBI864.csv


100%|██████████| 12915/12915 [00:02<00:00, 5224.90it/s]



=== NCBI876_xenium_nucleus_seg.parquet ===
Total cells: 12915
Classified cells: 12783 (99.0%)
Unknown cells (excluded): 132 (1.0%)

Cell type distribution (saved):
  Stromal: 9491 (74.2%)
  Neutrophil: 1619 (12.7%)
  Epithelial: 763 (6.0%)
  Plasma: 710 (5.6%)
  Lymphocyte: 200 (1.6%)
✅ SAVED: NCBI876.csv


100%|██████████| 23758/23758 [00:04<00:00, 5319.14it/s]


...

100%|██████████| 26396/26396 [00:08<00:00, 3285.92it/s]



=== NCBI859_xenium_nucleus_seg.parquet ===
Total cells: 26396
Classified cells: 26310 (99.7%)
Unknown cells (excluded): 79 (0.3%)

Cell type distribution (saved):
  Stromal: 24654 (93.7%)
  Plasma: 543 (2.1%)
  Epithelial: 508 (1.9%)
  Lymphocyte: 387 (1.5%)
  Neutrophil: 218 (0.8%)
✅ SAVED: NCBI859.csv


100%|██████████| 27973/27973 [00:07<00:00, 3656.72it/s]



=== NCBI873_xenium_nucleus_seg.parquet ===
Total cells: 27973
Classified cells: 27881 (99.7%)
Unknown cells (excluded): 90 (0.3%)

Cell type distribution (saved):
  Stromal: 25251 (90.6%)
  Epithelial: 1554 (5.6%)
  Plasma: 578 (2.1%)
  Neutrophil: 382 (1.4%)
  Lymphocyte: 116 (0.4%)
✅ SAVED: NCBI873.csv


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import openslide as ops

reduction_factor=20
slide=ops.OpenSlide('../../data/spatialTranscriptome/detail_preprocessed_xenium/wsis/NCBI785.tif')
thumbnail = slide.get_thumbnail((slide.level_dimensions[0][0] // reduction_factor, slide.level_dimensions[0][1] // reduction_factor))
mask=np.ones_like(np.array(thumbnail)) * 0
labels_df=pd.read_csv('../../data/spatialTranscriptome/detail_preprocessed_xenium/labels/NCBI785.csv')

fig, ax = plt.subplots(figsize=(22, 20))

for idx, row in labels_df.iterrows():
    x=row['x1']/reduction_factor + row['x2']/reduction_factor
    x=x//2
    y=row['y1']/reduction_factor + row['y2']/reduction_factor
    y=y//2
    mask[int(y):int(y)+2, int(x):int(x)+2]=np.array(class_colors[row['class_name']])/255.

ax.imshow(mask*0.5 + np.array(thumbnail)/255.*0.5)
ax.axis('off')
ax.set_title('Cell Type Annotation', fontsize=16, fontweight='bold')

# 클래스별 개수 계산
class_counts = labels_df['class_name'].value_counts()

# 범례 추가 (클래스 개수 포함)
legend_patches = []
for class_name, hex_color in class_colors_hex.items():
    count = class_counts.get(class_name, 0)
    label = f"{class_name}: {count}"
    patch = mpatches.Patch(color=hex_color, label=label)
    legend_patches.append(patch)

ax.legend(handles=legend_patches, 
         loc='upper right', 
         fontsize=15,
         framealpha=0.95,
         bbox_to_anchor=(1.18, 1.0),
         title='Cell Type (Count)',
         title_fontsize=12)

plt.tight_layout()
plt.show()

# 전체 통계 출력
print("=== Cell Type Statistics ===")
print(f"Total cells: {len(labels_df)}")
print("\nCell type distribution:")
print(class_counts.sort_index())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import openslide as ops

reduction_factor=20
slide=ops.OpenSlide('../../data/spatialTranscriptome/detail_preprocessed_xenium/wsis/TENX106.tif')
thumbnail = slide.get_thumbnail((slide.level_dimensions[0][0] // reduction_factor, slide.level_dimensions[0][1] // reduction_factor))
mask=np.ones_like(np.array(thumbnail)) * 0
labels_df=pd.read_csv('../../data/spatialTranscriptome/detail_preprocessed_xenium/labels/TENX106.csv')

fig, ax = plt.subplots(figsize=(22, 20))

for idx, row in labels_df.iterrows():
    x=row['x1']/reduction_factor + row['x2']/reduction_factor
    x=x//2
    y=row['y1']/reduction_factor + row['y2']/reduction_factor
    y=y//2
    
    mask[int(y):int(y)+2, int(x):int(x)+2]=np.array([class_colors[row['class_name']]])/255.

ax.imshow(mask*0.5 + np.array(thumbnail)/255.*0.5)
ax.axis('off')
ax.set_title('Cell Type Annotation', fontsize=16, fontweight='bold')

# 클래스별 개수 계산
class_counts = labels_df['class_name'].value_counts()

# 범례 추가 (클래스 개수 포함)
legend_patches = []
for class_name, hex_color in class_colors_hex.items():
    count = class_counts.get(class_name, 0)
    label = f"{class_name}: {count}"
    patch = mpatches.Patch(color=hex_color, label=label)
    legend_patches.append(patch)

ax.legend(handles=legend_patches, 
         loc='upper right', 
         fontsize=15,
         framealpha=0.95,
         bbox_to_anchor=(1.18, 1.0),
         title='Cell Type (Count)',
         title_fontsize=12)

plt.tight_layout()
plt.show()

# 전체 통계 출력
print("=== Cell Type Statistics ===")
print(f"Total cells: {len(labels_df)}")
print("\nCell type distribution:")
print(class_counts.sort_index())

In [ ]:
df_filtered[df_filtered['cell_id']==b'aaaeppaj-1']

In [ ]:
grouped_transcripts